# Sultai mana-base explorer

Run all cells once. Then edit **Dual penalty** and **Rainbow penalty** in the parameter cell and rerun the analysis cell below it. The expensive draw calculations stay cached in the tables.

Each penalty is charged per land in the base. The starting values are illustrative, not calibrated life costs. Modes have uniform weights within each physical spell profile.

Dual sources are capped at four each, each basic at five, and rainbow sources at eight. One-swap analysis runs by default; set `run_exhaustive = True` in the build cell to evaluate the full pool.


In [1]:
from pathlib import Path
import sys
from time import perf_counter
from html import escape

# Works when launched from the repository or one of its subdirectories.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_opportunities.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from IPython.display import HTML, display
from tqdm.auto import tqdm
from mana_opportunities import CastingMode, ManaSource, Spell, SpellCost
from mana_search import ManaSpace, build_component_table


def show_table(headers, rows):
    header = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def source_name(source):
    return f'{source.kind}:{source.colors}'


def swap_text(swap):
    if not swap.cards_replaced:
        return 'No replacement'
    out = ', '.join(f'{n} {source_name(s)}' for s, n in swap.out)
    incoming = ', '.join(f'{n} {source_name(s)}' for s, n in swap.incoming)
    return f'OUT {out}; IN {incoming}'


## Spell profiles and starting mana base

Edit these definitions here, then rerun this cell, build the search tables, and run the analysis. Counts in `MANA_BASE` describe the starting composition; dual search bounds are capped at four in the build cell. The total land count is inferred from this composition. Mode weights default to uniform within each spell profile.


In [2]:
SPELLS = (
    SpellCost("2UB", 4),  # Superior Spider-Man
    Spell(4, tuple(CastingMode(c) for c in ("UU", "BB", "4UU", "4BB", "2UUBB"))),  # Deceit
    Spell(4, tuple(CastingMode(c) for c in ("WW", "BB", "4WW", "4BB", "2WWBB"))),  # Emptiness
    Spell(4, tuple(CastingMode(c) for c in ("UU", "GG", "3UU", "3GG", "1UUGG"))),  # Wistfulness
    SpellCost("2G", 4),   # Formidable Speaker
    SpellCost("1B", 3),   # Bitter Triumph
    SpellCost("B", 4),    # Moonshadow
    SpellCost("7UU", 4),  # Sunderflock
    SpellCost("BGU", 3),  # Awaken the Honored Dead
)

MANA_BASE = {
    ManaSource("U", "basic"): 4,
    ManaSource("B", "basic"): 4,
    ManaSource("G", "basic"): 2,
    ManaSource("UB", "dual"): 4,
    ManaSource("UG", "dual"): 4,
    ManaSource("BG", "dual"): 4,
    ManaSource("WUBRG", "rainbow"): 4,
}


## Build search tables

Run this cell again only if you change the spell profile, bounds, or evaluation settings. The Sultai input uses an illustrative mana base with basic, dual, and rainbow sources; conditional lands and Cavern are excluded.


In [3]:
run_exhaustive = True  # Enable explicitly: the expanded pool can take much longer.

land_count = sum(MANA_BASE.values())
space = ManaSpace(
    {source: (0, {"basic": 5, "dual": 4, "rainbow": 8}[source.kind]) for source in MANA_BASE},
    land_count=land_count,
)
print(f'Exhaustive candidates: {space.candidate_count()}')
tables = {}
for mode in (('exhaustive', 'one_swap') if run_exhaustive else ('one_swap',)):
    started = perf_counter()
    total = space.candidate_count() if mode == 'exhaustive' else space.one_swap_count(MANA_BASE)
    tables[mode] = build_component_table(
        space, SPELLS, mode=mode, baseline=MANA_BASE,
        progress=lambda candidates: tqdm(candidates, total=total, desc=mode, unit='base'),
    )
    print(f'{mode}: {len(tables[mode].rows)} bases built in {perf_counter() - started:.2f}s')


Exhaustive candidates: 4122


exhaustive:   0%|          | 0/4122 [00:00<?, ?base/s]

exhaustive: 4122 bases built in 29.94s


one_swap:   0%|          | 0/25 [00:00<?, ?base/s]

one_swap: 25 bases built in 0.18s


## Parameters

Edit these two values, run this cell, then run the analysis cell. Zero disables that penalty. Changing these values does not require rebuilding the tables.


In [4]:
dual_penalty = 0.01
rainbow_penalty = 0.02


## Analysis

Shows the five highest scores and concrete replacements from the original base. Sensitivity intervals describe each candidate set's current winner while varying one penalty and holding the other fixed. Endpoints include ties; crossing beyond an endpoint selects the displayed replacement. Fractions are retained in the underlying result objects.


In [5]:
penalties = dict(dual_penalty=dual_penalty, rainbow_penalty=rainbow_penalty)
analyses = {}
for mode, table in tables.items():
    ranked = table.rank(**penalties)
    winner = dict(zip(space.sources, ranked[0].counts))
    display(HTML(f'<h3>{escape(mode)} ? {len(table.rows)} candidates</h3>'))
    print(f'Dual penalty: {dual_penalty}; rainbow penalty: {rainbow_penalty}')
    show_table(
        ['Rank', 'Raw opportunities', 'Penalty', 'Score', 'Gain from start', 'Replacements from start'],
        [(i, f'{float(r.raw_score):.6f}', f'{float(r.penalty):.6f}', f'{float(r.score):.6f}',
          f'{float(r.delta):+.6f}', swap_text(r.swap)) for i, r in enumerate(ranked[:5], 1)],
    )
    show_table(['Winning source', 'Count'], [(source_name(s), n) for s, n in winner.items() if n])
    sensitivities = {}
    rows = []
    for parameter in ('dual_penalty', 'rainbow_penalty'):
        sensitivity = table.crossings(parameter, baseline=winner, **penalties)
        sensitivities[parameter] = sensitivity
        interval = sensitivity.optimal_interval
        upper = 'unbounded' if interval.upper is None else f'{float(interval.upper):.8g}'
        rows.append((parameter, f'[{float(interval.lower):.8g}, {upper}]',
                     'none' if sensitivity.lower is None else swap_text(sensitivity.lower.swap),
                     'none' if sensitivity.upper is None else swap_text(sensitivity.upper.swap)))
    show_table(['Varied penalty', 'Winner stays optimal', 'Below lower boundary', 'Above upper boundary'], rows)
    analyses[mode] = dict(ranked=ranked, sensitivities=sensitivities)


Dual penalty: 0.01; rainbow penalty: 0.02


Rank,Raw opportunities,Penalty,Score,Gain from start,Replacements from start
1,3.222016,0.260000,2.962016,+0.076659,"OUT 1 basic:B, 1 basic:U, 2 dual:UG; IN 4 rainbow:WUBRG"
2,3.230834,0.270000,2.960834,+0.075477,"OUT 1 basic:B, 2 basic:U, 1 dual:UG; IN 4 rainbow:WUBRG"
3,3.230790,0.270000,2.960790,+0.075433,"OUT 1 basic:B, 1 basic:G, 1 basic:U, 1 dual:UG; IN 4 rainbow:WUBRG"
4,3.230773,0.270000,2.960773,+0.075416,"OUT 1 basic:G, 2 basic:U, 1 dual:UG; IN 4 rainbow:WUBRG"
5,3.210562,0.250000,2.960562,+0.075205,"OUT 1 basic:B, 1 basic:U, 3 dual:UG; IN 1 basic:G, 4 rainbow:WUBRG"


Winning source,Count
basic:B,3
basic:G,2
basic:U,3
dual:BG,4
dual:UB,4
dual:UG,2
rainbow:WUBRG,8


Varied penalty,Winner stays optimal,Below lower boundary,Above upper boundary
dual_penalty,"[0.008818549, 0.011453763]",OUT 1 basic:U; IN 1 dual:UG,OUT 1 dual:UG; IN 1 basic:G
rainbow_penalty,"[0, 0.03387023]",none,"OUT 1 basic:G, 1 basic:U, 1 rainbow:WUBRG; IN 1 basic:B, 2 dual:UG"


Dual penalty: 0.01; rainbow penalty: 0.02


Rank,Raw opportunities,Penalty,Score,Gain from start,Replacements from start
1,3.131671,0.220000,2.911671,+0.026314,OUT 1 basic:U; IN 1 rainbow:WUBRG
2,3.128321,0.220000,2.908321,+0.022964,OUT 1 basic:G; IN 1 rainbow:WUBRG
3,3.117588,0.210000,2.907588,+0.022231,OUT 1 dual:UG; IN 1 rainbow:WUBRG
4,3.124369,0.220000,2.904369,+0.019012,OUT 1 basic:B; IN 1 rainbow:WUBRG
5,3.111932,0.210000,2.901932,+0.016575,OUT 1 dual:UB; IN 1 rainbow:WUBRG


Winning source,Count
basic:B,4
basic:G,2
basic:U,3
dual:BG,4
dual:UB,4
dual:UG,4
rainbow:WUBRG,5


Varied penalty,Winner stays optimal,Below lower boundary,Above upper boundary
dual_penalty,"[0, 0.01408325]",none,OUT 1 dual:UG; IN 1 basic:U
rainbow_penalty,"[0, 0.042250553]",none,OUT 1 rainbow:WUBRG; IN 1 basic:B
